# Ключевая ставка ЦБ и валютные пары Рубль - Доллар/Юань
- Откуда грузить данные по котировкам валют -- https://www.cbr.ru/development/dws/ || https://www.cbr.ru/development/sxml/
- Справочники
- Единичные/разовые выгрузки
- Выгрузки за период

- Как грузить данные по ключевой ставке
- Истоник, варианты и формат выгрузки

In [2]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET

from datetime import datetime
from io import BytesIO

In [3]:
# Получение справочника валют

url_catalog = 'http://www.cbr.ru/scripts/XML_valFull.asp'
response_catalog = requests.get(
    url=url_catalog,
    timeout=30
)

print(response_catalog.url)
if response_catalog.status_code != 200:
    response_catalog.raise_for_status()

df_catalog = pd.read_xml(
    BytesIO(response_catalog.content),
    xpath=".//Item",
    parser="etree",
    encoding="windows-1251"
)
df_catalog.head()

http://www.cbr.ru/scripts/XML_valFull.asp


,ID,Name,EngName,Nominal,ParentCode,ISO_Num_Code,ISO_Char_Code
0,R01010,Австралийский доллар,Australian Dollar,1,R01010,36.0,AUD
1,R01015,Австрийский шиллинг,Austrian Shilling,1000,R01015,40.0,ATS
2,R01020A,Азербайджанский манат,Azerbaijan Manat,1,R01020,944.0,AZN
3,R01030,Алжирский динар,Algerian Dinar,100,R01030,12.0,DZD
4,R01035,Фунт стерлингов,Pound Sterling,1,R01035,826.0,GBP


In [4]:
currency_id = df_catalog[df_catalog['Name'].isin(['Юань', 'Доллар США'])]['ID'].unique().tolist()
currency_id

['R01235', 'R01375']

In [5]:
# Выгрузить данные по котировкам на текущую дату
today = datetime.today()

url_quotes_for_date= f'http://www.cbr.ru/scripts/XML_daily.asp?date_req={today.day:02d}/{today.month:02d}/{today.year}.asp'

params_quotes_for_date = {
    "date_req": today.strftime("%d/%m/%Y") # Передать дату альтернативно, через параметр
}

response_today = requests.get(
    url=url_quotes_for_date,
    timeout=30
)


print(response_today.url)
if response_today != 200:
    response_today.raise_for_status()

date = ET.fromstring(response_today.content).attrib["Date"]
print(f'Выгрузка котировки выгружены на: {date}')

df_foreign_currency = pd.read_xml(
    BytesIO(response_today.content),
    xpath=".//Valute",
    parser="etree",
    encoding="windows-1251"
)

df_foreign_currency[df_foreign_currency['ID'].isin(currency_id)]

http://www.cbr.ru/scripts/XML_daily.asp?date_req=12/08/2026.asp
Выгрузка котировки выгружены на: 12.08.2026


,ID,NumCode,CharCode,Nominal,Name,Value,VunitRate
15,R01235,840,USD,1,Доллар США,"82,3742","82,3742"
25,R01375,156,CNY,1,Юань,"12,1819","12,1819"


In [6]:
# Получение котировок в динамике

url_for_period = 'https://www.cbr.ru/scripts/XML_dynamic.asp'
params_for_period = {
    "date_req1": "01/01/2026",
    "date_req2": today.strftime("%d/%m/%Y"),
}

list_dynamic: list[pd.DataFrame] = []
for currency_code in currency_id:
    params_for_period['VAL_NM_RQ'] = currency_code
    response_period = requests.get(
        url=url_for_period,
        params = params_for_period,
        timeout=30
        )

    if response_period != 200:
        response_period.raise_for_status()

    _df = pd.read_xml(
        BytesIO(response_period.content),
        xpath=".//Record",
        parser="etree",
        encoding="windows-1251"
        )

    list_dynamic.append(_df)

df_dynamic = pd.concat(list_dynamic, ignore_index=True)
df_dynamic = pd.merge(df_dynamic, df_catalog[['ID', 'ISO_Char_Code']].drop_duplicates(),
                      left_on='Id', right_on='ID', how='left').drop(columns='ID')


In [7]:
df_dynamic.head()

,Date,Id,Nominal,Value,VunitRate,ISO_Char_Code
0,13.01.2026,R01235,1,"78,7913","78,7913",USD
1,14.01.2026,R01235,1,"78,8527","78,8527",USD
2,15.01.2026,R01235,1,"78,5711","78,5711",USD
3,16.01.2026,R01235,1,"78,5280","78,528",USD
4,17.01.2026,R01235,1,"77,8332","77,8332",USD


In [8]:
df_dynamic.tail()

,Date,Id,Nominal,Value,VunitRate,ISO_Char_Code
289,06.08.2026,R01375,1,"11,9684","11,9684",CNY
290,07.08.2026,R01375,1,"12,0637","12,0637",CNY
291,08.08.2026,R01375,1,"12,1655","12,1655",CNY
292,11.08.2026,R01375,1,"12,2546","12,2546",CNY
293,12.08.2026,R01375,1,"12,1819","12,1819",CNY


In [9]:
# Получить динамику ключевой ставки ЦБ с начала текущего года

from_date = today.replace(
    month=1,
    day=1
)

url_key_rate = "https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx"

headers_key_rate = {
    "Content-Type": "text/xml; charset=utf-8",
    "SOAPAction": "http://web.cbr.ru/KeyRateXML"
}

soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
    <soap:Body>
        <KeyRateXML xmlns="http://web.cbr.ru/">
            <fromDate>{from_date.strftime("%Y-%m-%dT00:00:00")}</fromDate>
            <ToDate>{today.strftime("%Y-%m-%dT00:00:00")}</ToDate>
        </KeyRateXML>
    </soap:Body>
</soap:Envelope>
"""

response_key_rate = requests.post(
    url=url_key_rate,
    headers=headers_key_rate,
    data=soap_body,
    timeout=30
)

response_key_rate.raise_for_status()

print(response_key_rate.status_code)
print(response_key_rate.text[:2000])

df_key_rate = pd.read_xml(
    BytesIO(response_key_rate.content),
    xpath=".//KR",
    parser="etree",
    encoding="utf-8"
)

df_key_rate["DT"] = pd.to_datetime(df_key_rate["DT"]).dt.date
df_key_rate.head()

200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><KeyRateXMLResponse xmlns="http://web.cbr.ru/"><KeyRateXMLResult><KeyRate xmlns=""><KR><DT>2026-08-12T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-08-11T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-08-10T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-08-07T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-08-06T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-08-05T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-08-04T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-08-03T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-07-31T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-07-30T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-07-29T00:00:00+03:00</DT><Rate>14.00</Rate></KR><KR><DT>2026-07-28T00:00

,DT,Rate
0,2026-08-12,14.0
1,2026-08-11,14.0
2,2026-08-10,14.0
3,2026-08-07,14.0
4,2026-08-06,14.0


In [10]:
df_key_rate.tail()

,DT,Rate
149,2026-01-12,16.0
150,2026-01-09,16.0
151,2026-01-08,16.0
152,2026-01-06,16.0
153,2026-01-05,16.0
